<a href="https://colab.research.google.com/github/Nawaf-Alorabi/Tw_ML_Predicting_Employee_Burnout_with_Regression/blob/main/project_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Phase One: Data Loading & Core Integration
### Overview:
The primary goal of this phase is to establish a unified foundation for our study.
Since our analysis spans both administrative HR data and behavioral mental health metrics,
we must perform a strategic integration. This phase ensures that our 'Master Dataset'
is structurally sound and free from data leakage before any predictive modeling begins.

### Key Objectives:
1. Structural Audit: Checking dimensions and data types.
2. Data Integrity: Ensuring 1-to-1 mapping during merge.

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os

In [8]:
# 1. Download IBM HR Analytics Dataset

ibm_path = kagglehub.dataset_download("aryanpatel212/ibm-hr-analytics-cleaned-and-merged-master")
print("Path to dataset files:", ibm_path)

Using Colab cache for faster access to the 'ibm-hr-analytics-cleaned-and-merged-master' dataset.


In [9]:
# 2. Download Employee Mental Health Dataset
# Download latest version
mental_path = kagglehub.dataset_download("suhanigupta04/employee-mental-health-and-burnout-dataset")
print("Path to dataset files:", mental_path)

Using Colab cache for faster access to the 'employee-mental-health-and-burnout-dataset' dataset.
Path to dataset files: /kaggle/input/employee-mental-health-and-burnout-dataset


In [ ]:
# Function to locate and load CSV files automatically from kagglehub paths
def load_csv_from_path(path, selected_cols):
    files = [f for f in os.listdir(path) if f.endswith('.csv')]
    if not files:
        raise FileNotFoundError(f"No CSV file found in {path}")
    final_path = os.path.join(path, files[0])
    return pd.read_csv(final_path, usecols=selected_cols)

In [ ]:
# Define columns
cols_ibm = ['Age', 'Gender', 'JobRole', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'DistanceFromHome']
cols_mental = ['gender', 'job_role', 'sleep_hours', 'work_hours_per_week', 'meetings_per_day', 'physical_activity_days']

# 3. Load Datasets using the dynamic paths
df_ibm = load_csv_from_path(ibm_path, cols_ibm)
df_mental = load_csv_from_path(mental_path, cols_mental)

In [ ]:
# Load only necessary columns
cols_ibm = ['Age', 'Gender', 'JobRole', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'DistanceFromHome']
cols_mental = ['gender', 'job_role', 'sleep_hours', 'work_hours_per_week', 'meetings_per_day', 'physical_activity_days']

df_ibm = pd.read_csv('HR_Analytics_Cleaned_Master.csv', usecols=cols_ibm)
df_mental = pd.read_csv('tech_mental_health_burnout.csv', usecols=cols_mental)

In [ ]:
# Take only the important columns
cols_ibm = ['Age', 'Gender', 'JobRole', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'DistanceFromHome']
cols_mental = ['gender', 'job_role', 'sleep_hours', 'work_hours_per_week', 'meetings_per_day', 'physical_activity_days']

# Load Dataset
df_ibm = pd.read_csv('HR_Analytics_Cleaned_Master.csv', usecols=cols_ibm)
df_mental = pd.read_csv('tech_mental_health_burnout.csv', usecols=cols_mental)

In [ ]:
# Unify labels (Mapping)
mapping = {
    'Research Scientist': 'Data Scientist',
    'Laboratory Technician': 'ML Engineer',
    'Manufacturing Director': 'Software Engineer',
    'Healthcare Representative': 'Backend Developer',
    'Sales Executive': 'Frontend Developer',
    'Sales Representative': 'DevOps'
}
df_ibm['JobRole'] = df_ibm['JobRole'].replace(mapping).str.lower()
df_ibm['Gender'] = df_ibm['Gender'].str.lower()
df_mental['job_role'] = df_mental['job_role'].str.lower()
df_mental['gender'] = df_mental['gender'].str.lower()

In [ ]:

# Assign temporary IDs to perform a 1-to-1 mapping within each group
df_ibm['temp_id'] = df_ibm.groupby(['JobRole', 'Gender']).cumcount()
df_mental['temp_id'] = df_mental.groupby(['job_role', 'gender']).cumcount()

# Perform the merge using the temporary IDs
merged_df = pd.merge(df_ibm, df_mental,
                     left_on=['JobRole', 'Gender', 'temp_id'],
                     right_on=['job_role', 'gender', 'temp_id'],
                     how='inner')

In [ ]:
# Drop temporary and redundant columns
merged_df.drop(['job_role', 'gender', 'temp_id'], axis=1, inplace=True)

# 4. Export the final dataset
merged_df.to_csv('Final_Burnout_Dataset.csv', index=False)
print(f"Merge successful! Total final rows: {len(merged_df)}")

Merge successful! Total final rows: 3635


In [ ]:
merged_df.shape

(3635, 11)

In [ ]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3635 entries, 0 to 3634
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     3635 non-null   int64  
 1   DistanceFromHome        3635 non-null   int64  
 2   Gender                  3635 non-null   object 
 3   JobRole                 3635 non-null   object 
 4   MonthlyIncome           3635 non-null   int64  
 5   TotalWorkingYears       3635 non-null   float64
 6   YearsAtCompany          3635 non-null   int64  
 7   work_hours_per_week     3635 non-null   float64
 8   meetings_per_day        3635 non-null   float64
 9   sleep_hours             3635 non-null   float64
 10  physical_activity_days  3635 non-null   float64
dtypes: float64(5), int64(4), object(2)
memory usage: 312.5+ KB


In [ ]:
merged_df.head()

,Age,DistanceFromHome,Gender,JobRole,MonthlyIncome,TotalWorkingYears,YearsAtCompany,work_hours_per_week,meetings_per_day,sleep_hours,physical_activity_days
0,51,6,female,backend developer,131160,1.0,1,45.0,5.0,5.9,2.0
1,31,10,female,data scientist,41890,6.0,5,41.0,9.0,5.0,1.0
2,32,17,male,frontend developer,193280,5.0,5,56.0,6.0,7.1,1.0
3,32,10,male,frontend developer,23420,9.0,6,45.0,4.0,6.9,5.0
4,28,11,male,frontend developer,58130,5.0,0,52.0,5.0,5.1,5.0


In [ ]:
# chaeck for missing value
merged_df.isnull().sum()

,0
Age,0
DistanceFromHome,0
Gender,0
JobRole,0
MonthlyIncome,0
TotalWorkingYears,0
YearsAtCompany,0
work_hours_per_week,0
meetings_per_day,0
sleep_hours,0


In [ ]:
# Checking for missing duplicates
merged_df.duplicated().sum()

np.int64(0)

In [ ]:
merged_df.describe()

,Age,DistanceFromHome,MonthlyIncome,TotalWorkingYears,YearsAtCompany,work_hours_per_week,meetings_per_day,sleep_hours,physical_activity_days
count,3635.000000,3635.000000,3635.000000,3635.000000,3635.000000,3635.000000,3635.000000,3635.000000,3635.000000
mean,36.932875,9.289684,65471.760660,11.208528,6.957359,47.014305,4.058872,6.491334,2.612655
std,9.164245,8.145518,47401.214976,7.727969,5.980301,7.864308,1.985320,1.208794,1.825688
min,18.000000,1.000000,10090.000000,0.000000,0.000000,30.000000,0.000000,3.000000,0.000000
25%,30.000000,2.000000,28990.000000,6.000000,3.000000,42.000000,3.000000,5.700000,1.000000
50%,36.000000,7.000000,49410.000000,10.000000,5.000000,47.000000,4.000000,6.500000,2.000000
75%,43.000000,14.000000,84740.000000,15.000000,10.000000,52.000000,5.000000,7.300000,4.000000
max,60.000000,29.000000,199990.000000,40.000000,40.000000,74.000000,10.000000,10.000000,7.000000


In [ ]:
# Summary of Categorical (Non-numeric) Columns
merged_df.describe(include='object')

,Gender,JobRole
count,3635,3635
unique,2,6
top,male,frontend developer
freq,2199,959
